# Algorithm 1 Refactor Validation

This notebook validates the refactored Algorithm 1 implementation with fixes for:

1. **Missing-day handling**: Reindex to full trading calendar
2. **Trading calendar**: Use equity_prices.index instead of freq='B'
3. **Split adjustment**: REMOVED (option prices already split-adjusted)
4. **PFD flexibility**: Search ±3 days for call/put pair
5. **Delta filtering**: Only require delta at PFD for weights

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import warnings
warnings.filterwarnings('ignore')

# Add src_refactor to path
sys.path.insert(0, '.')

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

NumPy: 2.4.2
Pandas: 2.2.3


In [2]:
# Import refactored module (NOT the original src/)
from src_refactor import (
    process_single_asset_v2,
    build_straddle_dataset_v2,
    get_portfolio_formation_days,
)

print("Refactored modules loaded from src_refactor/")

Refactored modules loaded from src_refactor/


## Load Data

In [3]:
from pathlib import Path
import pyarrow.parquet as pq

# Configuration
RAW_DATA_DIR = Path('raw_data')
FILTERED_DIR = RAW_DATA_DIR / 'options_filtered_by_year'
START_YEAR = 2011
END_YEAR = 2023

# Load security names
secnames_df = pd.read_parquet(RAW_DATA_DIR / 'security_names.parquet')
secid_to_ticker = dict(zip(secnames_df['secid'].astype(int), secnames_df['ticker']))
print(f"Loaded {len(secid_to_ticker)} security mappings")

# Only load columns needed
REQUIRED_COLUMNS = [
    'secid', 'date', 'exdate', 'cp_flag', 'strike_price',
    'best_bid', 'best_offer', 'delta', 'spot_price'
]

# Load options data
options_dfs = []
for year in range(START_YEAR, END_YEAR + 1):
    fpath = FILTERED_DIR / f'options_filtered_{year}.parquet'
    if fpath.exists():
        table = pq.read_table(fpath, columns=REQUIRED_COLUMNS)
        df = table.to_pandas()
        options_dfs.append(df)
        print(f"  {year}: {len(df):,} records")

options_df = pd.concat(options_dfs, ignore_index=True)
print(f"\nTotal: {len(options_df):,} records")

Loaded 87 security mappings
  2011: 1,452,407 records
  2012: 1,665,902 records
  2013: 2,596,040 records
  2014: 4,226,163 records
  2015: 5,084,197 records
  2016: 5,158,134 records
  2017: 5,536,621 records
  2018: 6,153,349 records
  2019: 6,376,102 records
  2020: 6,360,620 records
  2021: 6,687,680 records
  2022: 5,938,605 records
  2023: 5,929,889 records

Total: 63,165,709 records


In [4]:
# Adapt data format
options_df['ticker'] = options_df['secid'].map(secid_to_ticker)
options_df = options_df.rename(columns={'strike_price': 'strike'})

# Drop rows without ticker mapping
options_df = options_df.dropna(subset=['ticker'])

# Convert to efficient dtypes
options_df['ticker'] = options_df['ticker'].astype('category')
options_df['cp_flag'] = options_df['cp_flag'].astype('category')
for col in ['best_bid', 'best_offer', 'delta', 'spot_price', 'strike']:
    options_df[col] = options_df[col].astype('float32')

print(f"Prepared {len(options_df):,} options records")
print(f"Tickers: {options_df['ticker'].nunique()}")

Prepared 63,165,709 options records
Tickers: 87


In [5]:
# Build equity price matrix from options data
equity_df = options_df.groupby(['date', 'ticker'])['spot_price'].first().unstack()
equity_df = equity_df.sort_index()

print(f"Equity price matrix: {equity_df.shape}")
print(f"  Trading days: {len(equity_df)}")
print(f"  Date range: {equity_df.index.min().date()} to {equity_df.index.max().date()}")

Equity price matrix: (3270, 87)
  Trading days: 3270
  Date range: 2011-01-03 to 2023-12-29


## Validation Test: 3 Tickers

In [6]:
# Test with 3 tickers: AAPL, MSFT, JPM
TEST_TICKERS = ['AAPL', 'MSFT', 'JPM']

# Filter data
test_options = options_df[options_df['ticker'].isin(TEST_TICKERS)].copy()
test_equity = equity_df[TEST_TICKERS].copy()

print(f"Test data:")
print(f"  Options: {len(test_options):,} records")
print(f"  Equity shape: {test_equity.shape}")

Test data:
  Options: 3,311,336 records
  Equity shape: (3270, 3)


In [ ]:
# Run refactored Algorithm 1
print("Running Refactored Algorithm 1...")
print("="*60)

MIN_COVERAGE = 0.90

try:
    straddle_prices, straddle_returns, metadata = build_straddle_dataset_v2(
        options_df=test_options,
        equity_df=test_equity,
        min_coverage=MIN_COVERAGE,
        verbose=True,
        return_metadata=True,
        debug=True
    )
    
    print(f"\nSUCCESS!")
    print(f"  Straddle prices shape: {straddle_prices.shape}")
    print(f"  Assets passing filter: {list(straddle_prices.columns)}")
    
except ValueError as e:
    print(f"\nFAILED: {e}")

Running Refactored Algorithm 1...


In [ ]:
# Detailed coverage analysis per ticker
print("\nDetailed Coverage Analysis:")
print("="*60)

pfds = get_portfolio_formation_days(
    test_equity.index.min().strftime('%Y-%m-%d'),
    test_equity.index.max().strftime('%Y-%m-%d')
)

for ticker in TEST_TICKERS:
    print(f"\n{ticker}:")
    
    prices, returns, meta = process_single_asset_v2(
        test_options,
        test_equity,
        ticker,
        pfds,
        min_coverage=0.0,  # Don't filter - we want to see coverage
        return_metadata=True,
        debug=True
    )
    
    if prices is not None:
        # Calculate coverage
        full_calendar = test_equity.loc[
            (test_equity.index >= pd.Timestamp(pfds[0])) &
            (test_equity.index <= pd.Timestamp(pfds[-1]))
        ].index
        
        expected = len(full_calendar)
        actual = prices.notna().sum()
        coverage = actual / expected
        
        print(f"  Expected days: {expected}")
        print(f"  Actual days:   {actual}")
        print(f"  Coverage:      {coverage:.1%}")
        print(f"  Passes 95%:    {'YES' if coverage >= 0.95 else 'NO'}")

## Visual Validation: AAPL Around Split Date

In [ ]:
# Plot AAPL straddle price around 2014 split (June 9, 2014)
if 'AAPL' in straddle_prices.columns:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # Full price series
    ax1 = axes[0]
    straddle_prices['AAPL'].plot(ax=ax1, label='AAPL Straddle Price')
    ax1.axvline(pd.Timestamp('2014-06-09'), color='red', linestyle='--', label='7:1 Split')
    ax1.axvline(pd.Timestamp('2020-08-31'), color='orange', linestyle='--', label='4:1 Split')
    ax1.set_title('AAPL Delta-Neutral Straddle Price (Full Series)')
    ax1.set_ylabel('Price ($)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Zoomed around 2014 split
    ax2 = axes[1]
    zoom = straddle_prices['AAPL'].loc['2014-05':'2014-07']
    zoom.plot(ax=ax2, marker='o', markersize=3, label='AAPL Straddle Price')
    ax2.axvline(pd.Timestamp('2014-06-09'), color='red', linestyle='--', label='7:1 Split Date')
    ax2.set_title('AAPL Straddle Price Around 2014 Split (Zoomed)')
    ax2.set_ylabel('Price ($)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Check for discontinuity
    pre_split = straddle_prices['AAPL'].loc['2014-06-06':'2014-06-06']
    post_split = straddle_prices['AAPL'].loc['2014-06-09':'2014-06-09']
    
    if len(pre_split) > 0 and len(post_split) > 0:
        print(f"\nSplit continuity check:")
        print(f"  Pre-split (2014-06-06):  ${pre_split.values[0]:.2f}")
        print(f"  Post-split (2014-06-09): ${post_split.values[0]:.2f}")
        pct_change = (post_split.values[0] - pre_split.values[0]) / pre_split.values[0] * 100
        print(f"  Change: {pct_change:.1f}%")
        print(f"  Normal: {'YES' if abs(pct_change) < 20 else 'NO (possible discontinuity)'}")
else:
    print("AAPL not in results - check coverage issues")

## Run Full Pipeline (All 87 Tickers)

In [ ]:
# Run on ALL tickers
print("Running Full Algorithm 1 on All 87 Tickers...")
print("="*60)

try:
    full_prices, full_returns, full_meta = build_straddle_dataset_v2(
        options_df=options_df,
        equity_df=equity_df,
        min_coverage=MIN_COVERAGE,
        verbose=True,
        return_metadata=False,
        debug=False
    )
    
    print(f"\n" + "="*60)
    print(f"FULL PIPELINE SUCCESS!")
    print(f"  Straddle prices shape: {full_prices.shape}")
    print(f"  Trading days (delta): {full_prices.shape[0]}")
    print(f"  Assets (N): {full_prices.shape[1]}")
    print(f"  Assets: {list(full_prices.columns)}")
    
except ValueError as e:
    print(f"\nFailed: {e}")

## Summary

In [ ]:
print("="*60)
print("REFACTORED ALGORITHM 1 - VALIDATION SUMMARY")
print("="*60)

print("\nFIXES APPLIED:")
print("  1. Missing-day handling: Reindex to trading calendar [FIXED]")
print("  2. Trading calendar: Use equity index not freq='B' [FIXED]")
print("  3. Split adjustment: REMOVED (prices already adjusted) [FIXED]")
print("  4. PFD flexibility: +/-3 day lookahead [FIXED]")
print("  5. Delta filtering: Only at PFD [FIXED]")

if 'full_prices' in dir():
    print(f"\nRESULTS:")
    print(f"  Assets passing 95% filter: {len(full_prices.columns)}/87")
    print(f"  Trading days: {len(full_prices)}")
    print(f"  Date range: {full_prices.index.min().date()} to {full_prices.index.max().date()}")
    
    # Return statistics
    print(f"\nReturn Statistics:")
    print(f"  Mean daily return: {full_returns.mean().mean():.6f}")
    print(f"  Std daily return:  {full_returns.std().mean():.6f}")